## 🔹 1. Partitioning in Pyspark
### ✅ What is Partitioning?
Partitioning splits data into **separate directories based on column values** when writing data (usually in Hive-style format).

👉 Example:
```python
df.write.partitionBy("country").parquet("output/")
```
This creates:
```
output/
  country=India/
  country=USA/
  country=UK/
```


### ✅ Key Characteristics
* Done at **write time**
* Creates **directory structure**
* Works with file formats like Parquet/ORC
* Enables **partition pruning** (huge performance boost)

### ✅ Benefits
* Faster queries when filtering:
```python
df.filter("country = 'India'")
```
→ Only reads `country=India` folder
* Reduces I/O (big performance gain)


### ❌ Drawbacks
* Too many unique values → too many small files (bad)
* Not useful for high-cardinality columns (e.g., user_id)

### ✅ Best Use Cases
* Columns with **low to medium cardinality**
  * country
  * date
  * region

In [1]:
# importing libs
from pyspark.sql import SparkSession
from pyspark.sql import HiveContext

In [2]:
# creating the spark object
spark = SparkSession.Builder().master("local[*]").appName("partVsBuckeing").enableHiveSupport().getOrCreate()

In [3]:
# creating the DF
df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("airbnb.csv")

In [4]:
df.show(10)

+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|  -122.23|   37.88|              41.0|      880.0|         129.0|     322.0|     126.0|       8.3252|          452600.0|       NEAR BAY|
|  -122.22|   37.86|              21.0|     7099.0|        1106.0|    2401.0|    1138.0|       8.3014|          358500.0|       NEAR BAY|
|  -122.24|   37.85|              52.0|     1467.0|         190.0|     496.0|     177.0|       7.2574|          352100.0|       NEAR BAY|
|  -122.25|   37.85|              52.0|     1274.0|         235.0|     558.0|     219.0|       5.6431|          341300.0|       NEAR BAY|
|  -122.25|   37.85|              

In [5]:
# PARTITIONING
df.write.mode("overwrite").partitionBy("ocean_proximity").csv("airbnb_DB")

In [6]:
# cleanup the Database
from shutil import rmtree
import os
if os.path.exists("airbnb_DB"):
    rmtree("airbnb_DB")
    print("cleanup done...")
else:
    print("path not found...")

cleanup done...


## 🔹 2. Bucketing in PySpark

### ✅ What is Bucketing?
Bucketing distributes data into a fixed number of files using a **hash function on a column**.

👉 Example:

```python
df.write.bucketBy(4, "user_id").saveAsTable("users_bucketed")
```

This creates:
* 4 buckets (files)
* Data distributed based on hash(user_id)

### ✅ Key Characteristics
* Done at **write time**
* Does NOT create directories (unlike partitioning)
* Requires saving as a **table** (Hive support)

### ✅ Benefits
* Efficient **joins** between bucketed tables
* Avoids full shuffle if both tables are bucketed on same column
* Useful for **large datasets**

### ❌ Drawbacks
* Less flexible than partitioning
* Needs careful planning (fixed number of buckets)
* Not useful for filtering like partitioning


### ✅ Best Use Cases
* Large tables with frequent **joins**
* High-cardinality columns
  * user_id
  * transaction_id


## 🔥 Partitioning vs Bucketing (Important Interview Table)

| Feature           | Partitioning     | Bucketing                 |
| ----------------- | ---------------- | ------------------------- |
| Storage           | Directory-based  | File-based (inside table) |
| Trigger           | Write time       | Write time                |
| Column Type       | Low cardinality  | High cardinality          |
| Performance Use   | Filtering        | Joins                     |
| File Structure    | Multiple folders | Fixed number of files     |
| Shuffle Reduction | ❌ No             | ✅ Yes (for joins)         |


In [7]:
# BUCKETING
df.write.mode("overwrite").bucketBy(4, "ocean_proximity").saveAsTable("airbnb_DB")

In [8]:
spark.sql("drop table airbnb_DB")

DataFrame[]

In [ ]:
# Closing the all connections for cleanup
spark.stop()
import os
os._exit(0)

: 

In [1]:
# cleanup the Database
from shutil import rmtree
import os
if (os.path.exists("metastore_db") or os.path.exists("spark-warehouse")):
    rmtree("metastore_db", True)
    rmtree("spark-warehouse", True)
    print("cleanup done...")
else:
    print("path not found...")

cleanup done...



### 3. When to Use What?
* Use Partitioning when: 👉 Queries filter on a column
* Use Bucketing when: 👉 Frequent joins on a column

### 4. Important Interview Tips
* Partitioning = **horizontal data separation**
* Bucketing = **hash-based distribution**
* Too many partitions = **small file problem**
* Bucketing helps reduce **shuffle in joins**
* Spark may not always enforce bucketing unless:
  * `spark.sql.bucketing.enabled = true`

### 🔚 Summary
* **Partitioning → filtering optimization**
* **Bucketing → join optimization**
* Use both strategically for large-scale data processing